In [0]:
from dataclasses import dataclass
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import *
from delta.tables import *

In [0]:
# Create class for Bronze Layer
@dataclass
class BronzeLayer:
    file_path:str
    header:bool
    delimiter:str
    table_name:str

    def __post_init__(self) -> None:
        self.format_type = self.file_path.split('.')[-1]
        self.target_table_bronze = f'{self.table_name}'

    @classmethod
    def from_config_table(cls, pipeline_name:str) -> "BronzeLayer":
        conf = (spark.table("netflix.config_table")
               .filter(col("pipeline_name") == pipeline_name)
               .select("file_path", "header", "delimiter", "table_name")
               .first())
        return cls(
            file_path = conf.file_path
            , header = conf.header
            , delimiter = conf.delimiter
            , table_name = conf.table_name
        )
    def read_from_file(self) -> DataFrame:
        df = (
            spark.read.format(self.format_type)
            .option("header", self.header)
            .option("delimiter", self.delimiter)
            .load(self.file_path)
        )
        # return with another metadata columns
        return (
            df
            .withColumn("_load_dt", current_date())
            .withColumn("_load_dttm", current_timestamp())
            .withColumn("_file_name", col("_metadata.file_name"))
            .withColumn("_file_path", col("_metadata.file_path"))
            .withColumn("_file_size", col("_metadata.file_size"))
            .withColumn("_file_mod", col("_metadata.file_modification_time"))
        )
    def load_to_bronze_table(self, raw_df: DataFrame) -> None:
        # Ensure the bronze table is initialized with correct settings before loading
        self._init_bronze_table() 

        # write data to bronze table
        raw_df.write.mode("append").saveAsTable(self.target_table_bronze)
        print(f"Table {self.target_table_bronze} loaded")

    def _init_bronze_table(self) -> None:
        # created schema for first time and enable CDF to table
        ## 1. First check if table exists or not
        if spark.catalog.tableExists(self.target_table_bronze):
            print(f"Table {self.target_table_bronze} already exists.")
            return # end method immediately
            
        # 2. If table not exitsts. We create table
        else:
            print(f"Table {self.target_table_bronze} does not exist. Initializing...")
            
            # Retrive schema table (read with out data limit 0)
            source_df = self.read_from_file().limit(0)
            
            # Create table from Schema and enable CDF with Python API
            (source_df.write
             .format("delta")
             .option("delta.enableChangeDataFeed", "true")
             .mode("ignore") # prevent other class creating table before this action
             .saveAsTable(self.target_table_bronze))
            
            print(f"Table {self.target_table_bronze} created successfully with CDF enabled.")

In [0]:
# spark.table("workspace.netflix.config_table").display()

In [0]:
# %sql
# truncate table workspace.netflix.netflix_bronze

In [0]:
# Set current schema to netflix to ensure table is created in the correct location
# spark.sql("USE netflix")

# b = BronzeLayer.from_config_table("netflix")
# bronze_df = b.read_from_file()
# b.load_to_bronze_table(bronze_df)

# spark.table("workspace.netflix.netflix_bronze").display()

In [0]:
# %sql
# describe extended  workspace.netflix.netflix_bronze

In [0]:
# spark.table("workspace.netflix.netflix_bronze").display()

In [0]:
# %sql
# DESCRIBE HISTORY workspace.netflix.netflix_bronze

ต้องทำการสำรวจข้อมูลก่อน(ทำแล้ว) โดยเราต้องมีการทำความสะอาดข้อมูล ทำ Normalization Standardization ประมาณนี้
📋 ลำดับการ Clean Data ที่จะใช้ในโปรเจคนี้
    1. Load data from Bronze by use feature like CDF for get only new data(by compare current version and last version of data which processed already in control table) and Add serogete key to table # successes
    2. Trim String Column & Change Data Type
        1.1 Standardize Date & Business 
    3. Get Invalid Record (Data Quality Audit)
    4. Get Key Duplicate (Deduplication Logic)
    5. Get Row Duplicate
    6. Get only bad record and save to bad record table
    7. After get bad record left anti join for get final record and save it into Silver layer.


In [0]:
# create class for silver layer
@dataclass
class SilverLayer:
    table_name: str
    schema_detail: dict[str, str]
    keys: list[str]
    write_mode: str

    def __post_init__(self) -> None:
        self.bronze_table_name = f"{self.table_name}"
        self.silver_table_name = f"{self.table_name}"
        self.bad_record_table_name = f"{self.table_name}_bad_record"
        self.data_col = [col_name for col_name in self.schema_detail.keys()]
        self.invalid_rule = {"int": "^[0-9]+$", "date": "^\\d{4}-\\d{2}-\\d{2}$"}

    @classmethod
    def from_config_table(cls, pipeline_name: str) -> "SilverLayer":
        conf = (
            spark.table("workspace.netflix.config_table")
            .filter(col("pipeline_name") == pipeline_name)
            .select(
                "table_name", "schema_detail", "keys", "write_mode"
            )
            .first()
        )
        return cls(
            table_name=conf.table_name,
            schema_detail=conf.schema_detail,
            keys=conf.keys,
            write_mode=conf.write_mode,
        )
    # Helper Method for get invalid reason
    def _get_reason(self, df: DataFrame) -> DataFrame:
        control_col = [col_name for col_name in df.columns if col_name.startswith("_") and col_name != "_sk"]
        data_col = [col_name for col_name in df.columns if col_name.startswith("_")]
        or_statement = " OR ".join([col_name for col_name in control_col])
        return (
            df
            .filter(or_statement)
            .melt(
                ids = [*data_col, "_sk"]
                , values = control_col
                , variableColumnName = "reason"
                , valueColumnName = "status"
            )
        .filter(col("status") == True)
        .groupBy(*data_col, "_sk")
        .agg(collect_list("reason").alias("reason"))
        )
    
    def trim_data(self, df: DataFrame) -> DataFrame:
        """
        Trim all string columns to remove leading/trailing whitespace.
        Uses select() to avoid performance issues from .withColumn() in a loop.
        """
        # Pre-compute schema to avoid repeated Analyze RPCs
        df_columns = df.columns
        
        trim_exprs = [
            trim(col(col_name)).alias(col_name) if col_type == "string" else col(col_name)
            for col_name, col_type in self.schema_detail.items()
        ]
        # Include _sk if it exists
        if "_sk" in df_columns:
            trim_exprs.append(col("_sk"))
        
        return df.select(*trim_exprs)
    
    def change_data_type(self, df: DataFrame) -> DataFrame:
        """
        Change data type of columns based on schema_detail.
        For date columns, uses try_to_date() with "MMMM dd, yyyy" format.
        For other columns, uses try_cast() to return NULL for invalid conversions.
        Records with NULL values can be caught later in get_invalid_record().
        """
        # Pre-compute columns to avoid repeated Analyze RPCs
        df_columns = df.columns
        
        change_type_exprs = []
        
        for col_name, col_type in self.schema_detail.items():
            # Generic date handling - assumes "MMMM dd, yyyy" format for all date columns
            if col_type == "date":
                change_type_exprs.append(
                    expr(f"try_to_date({col_name}, 'MMMM dd, yyyy')").alias(col_name)
                )
            else:
                change_type_exprs.append(
                    expr(f"try_cast({col_name} as {col_type})").alias(col_name)
                )
        
        # Include _sk if it exists (using pre-computed df_columns)
        if "_sk" in df_columns:
            change_type_exprs.append(col("_sk"))
        
        return df.select(*change_type_exprs)
    
    def get_invalid_record(self, bronze_df: DataFrame) -> DataFrame:
        invalid_col = {
            f"_is_{col_name}_invalid": coalesce(~col(col_name).rlike(self.invalid_rule[col_type]), lit(False))
            for col_name, col_type in self.schema_detail.items() if col_type in ["int", "date"]
        }
        
        return (
            bronze_df
            .withColumns(invalid_col)
            .transform(self._get_reason)
        )
    
    def get_key_null_record(self, bronze_df: DataFrame) -> DataFrame:
        key_null_statement = { f"_is_{col_name}_null": col(col_name).isNull() for col_name in self.keys}

        return (
            bronze_df.withColumns(key_null_statement)
            .transform(self._get_reason)
        )
    
    def get_dup_record(self, df: DataFrame, key_null_df: DataFrame) -> DataFrame:
        pass # I will create for next day 14/06/2026
    
    def get_all_bad_record(self, invalid_df: DataFrame, key_null_df: DataFrame, duplicate_df: DataFrame) -> DataFrame:
        pass # I will create for next day 15/06/2026

    def get_final_result(self, df: DataFrame, bad_record_df: DataFrame) -> DataFrame:
        pass # I will create for next day 16/06/2026

    def load_bad_record(self, bad_record_df: DataFrame) -> None:
        pass # I will create for next day 17/06/2026

    def load_to_silver_layer(self, final_result_df: DataFrame, batch_id: int) -> None:
        """
        Load final results to silver table.
        batch_id: The batch number from foreachBatch (for logging/debugging).
        """
        pass # I will create for next day 18/06/2026


    def process_cdf_stream_to_silver(self, checkpoint_location: str = None) -> None:
        """
        Process CDF stream from bronze to silver with quality checks.
        Uses trigger(availableNow=True) for serverless-friendly incremental processing.
        """
        # Use provided checkpoint location or default to table-specific path
        if checkpoint_location is None:
            checkpoint_location = f"/Volumes/workspace/netflix/checkpoint_dir/{self.table_name}_silver/"
            
        # Create a stream from the bronze table
        cdf_stream = (
            spark.readStream
            .option("readChangeFeed", "true")
            .option("startingVersion", 0)  # Start from version 0 or use checkpoint
            .table(self.bronze_table_name)
            .filter(col("_change_type").isin(["insert", "update_postimage"]))
            .select(*self.data_col)
            .withColumn("_sk", monotonically_increasing_id())
        )

        query = (
            cdf_stream.writeStream
            .foreachBatch(self._process_quality_checks_batch)
            .option("checkpointLocation", checkpoint_location)
            .outputMode("update") # Only use when we use foreachBatch 
            .trigger(availableNow=True)
            .start()
        )
    
        query.awaitTermination()
        print(f"Stream processing complete for {self.silver_table_name}")

    def _process_quality_checks_batch(self, batch_df: DataFrame, batch_id: int) -> None:
        """
        Process each micro-batch through quality checks.
        Called automatically by foreachBatch with (batch_df, batch_id).
        """
        if batch_df.isEmpty():
            #print(f"Batch {batch_id}: No new records to process")
            return
        
        #print(f"Batch {batch_id}: Processing {batch_df.count()} records")
        
        # Standardize and clean DataFrame
        trimmed_stream = self.trim_data(batch_df)
        change_data_type_stream = self.change_data_type(trimmed_stream)
        invalid_df = self.get_invalid_record(change_data_type_stream)
        key_null_df = self.get_key_null_record(change_data_type_stream)
        duplicate_df = self.get_dup_record(change_data_type_stream, key_null_df)
        all_bad_df = self.get_all_bad_record(invalid_df, key_null_df, duplicate_df)
        final_df = self.get_final_result(change_data_type_stream, all_bad_df)

        # Load DataFrame into 2 tables: Silver table and Bad_record table
        self.load_bad_record(all_bad_df)
        self.load_to_silver_layer(final_df, batch_id)
        
        print(f"Batch {batch_id}: Complete")


In [0]:
# spark.table("workspace.netflix.netflix_bronze").display()

In [0]:
# spark.table("workspace.netflix.config_table").display()

In [0]:
# # Test all methods in one cell
# print("=" * 50)
# print("TESTING SILVER LAYER METHODS")
# print("=" * 50)

# # 1. Initialize SilverLayer
# print("\n1. Initialize SilverLayer")
# silver = SilverLayer.from_config_table("netflix")
# print(f"   Table: {silver.table_name}")
# print(f"   Keys: {silver.keys}")

# # 2. Load test data
# print("\n2. Load test data from bronze")
# test_df = (
#     spark.table(silver.bronze_table_name)
#     .limit(10)
#     .select(*silver.data_col)
#     .withColumn("_sk", monotonically_increasing_id())
# )
# print(f"   Loaded: {test_df.count()} records")
# print("\nOriginal Data:")
# display(test_df)

# # 3. Test trim_data
# print("\n3. Test trim_data method")
# trimmed_df = silver.trim_data(test_df)
# print("   ✅ trim_data completed")

# # 4. Test change_data_type
# print("\n4. Test change_data_type method")
# typed_df = silver.change_data_type(trimmed_df)
# print(f"   Schema after type conversion: {typed_df.dtypes}")
# print("\nAfter Type Conversion:")
# display(typed_df)

# # 5. Test get_invalid_record
# print("\n5. Test get_invalid_record method")
# invalid_df = silver.get_invalid_record(typed_df)
# print(f"   Invalid records found: {invalid_df.count()}")
# if invalid_df.count() > 0:
#     print("\nInvalid Records:")
#     display(invalid_df)
# else:
#     print("   ✅ No invalid records found")

# # 6. Test get_key_null_record
# print("\n6. Test get_key_null_record method")
# key_null_df = silver.get_key_null_record(typed_df)
# print(f"   Key null records found: {key_null_df.count()}")
# if key_null_df.count() > 0:
#     print("\nKey Null Records:")
#     display(key_null_df)
# else:
#     print("   ✅ No null key records found")

# print("\n" + "=" * 50)
# print("ALL TESTS COMPLETED SUCCESSFULLY! ✅")
# print("=" * 50)